In [109]:
import pandas as pd
import plotly.express as px


## Análise explotarória de dados

### Obervando os dados

In [110]:
# Importando os dados
car_data = pd.read_csv('vehicles_us.csv', sep=",") # lendo os dados

# Visualizando informações do dataframe
car_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 51525 entries, 0 to 51524
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   price         51525 non-null  int64  
 1   model_year    47906 non-null  float64
 2   model         51525 non-null  str    
 3   condition     51525 non-null  str    
 4   cylinders     46265 non-null  float64
 5   fuel          51525 non-null  str    
 6   odometer      43633 non-null  float64
 7   transmission  51525 non-null  str    
 8   type          51525 non-null  str    
 9   paint_color   42258 non-null  str    
 10  is_4wd        25572 non-null  float64
 11  date_posted   51525 non-null  str    
 12  days_listed   51525 non-null  int64  
dtypes: float64(4), int64(2), str(7)
memory usage: 7.7 MB


In [111]:
# Verificando a existência de duplicatas exatas
print("Duplicatas: ",car_data.duplicated().sum())

Duplicatas:  0


In [112]:
# Entendendo o que tem no campo com mais dados ausentes 'is_4wd'
print(car_data['is_4wd'].unique())

[ 1. nan]


In [113]:
# Entendendo a formatação dos nomes das cores em 'paint_color'
print(car_data['paint_color'].unique())

<ArrowStringArray>
[     nan,  'white',    'red',  'black',   'blue',   'grey', 'silver',
 'custom', 'orange', 'yellow',  'brown',  'green', 'purple']
Length: 13, dtype: str


In [114]:
# Entendendo as categorias de 'condition'
print(car_data['condition'].unique())

<ArrowStringArray>
['good', 'like new', 'fair', 'excellent', 'salvage', 'new']
Length: 6, dtype: str


In [115]:
# Entendendo o formato da data em 'date_posted'
print(car_data['date_posted'].head())

0    2018-06-23
1    2018-10-19
2    2019-02-07
3    2019-03-22
4    2019-04-02
Name: date_posted, dtype: str


Nas células acima é possivel compreender que:
- Não possui duplicatas
- os dados de 'is_4wd' são dados categóricos sim (1) ou não (NaN), onde será necessária a substituição dos ausentes por 0;
- os ausentes de 'paint_color' podem ser substituidos por 'unknown'
- os ausentes na quilometragem  'odometer' podem ser substituidos pela mediana agrupado por modelo e condição
- os ausentes em 'cylinders' e 'model_year' poedem ser ubstituidos pela mediana agrupado por modelo 
- date_posted precisa modificar o tipo para data no formato ano-mes-dia

### Modificando os dados

In [116]:
# Modificando ausentes de 'is_4wd' como 0 e passando para inteiro
car_data['is_4wd'] = car_data['is_4wd'].fillna(0).astype(int)

In [ ]:
# Substituindo os ausentes de 'paint_color' para unknown
car_data['paint_color'] = car_data['paint_color'].fillna('unknown')

SyntaxError: invalid syntax (4245050211.py, line 5)

In [ ]:
# Transformando os dados ausentes para quilometragem pela mediana para modelo e condição do carro
car_data['odometer'] = car_data['odometer'].fillna(
    car_data.groupby(['model', 'condition'])['odometer'].transform('median')
)

# Caso ainda existam nulos, transformando os ausentes com base apenas na mediana dos modelos
car_data['odometer'] = car_data['odometer'].fillna(
    car_data.groupby('model')['odometer'].transform('median')
)


In [ ]:
# Substituindo os ausentes em cylinders pela mediana por modelos e passando para int
car_data['cylinders'] = car_data['cylinders'].fillna(
    car_data.groupby('model')['cylinders'].transform('median')
).astype(int)

In [ ]:
# Substituindo os ausentes em model_year pela mediana por modelos e passando para int
car_data['model_year'] = car_data['model_year'].fillna(
    car_data.groupby('model')['model_year'].transform('median')
).astype(int)

In [ ]:
# Mudando o fromato de date_posted para data no formato ano-mes-dia
car_data['date_posted'] = pd.to_datetime(car_data['date_posted'])

# Verificando o formato
print(car_data['date_posted'].head)

<bound method NDFrame.head of 0       2018-06-23
1       2018-10-19
2       2019-02-07
3       2019-03-22
4       2019-04-02
           ...    
51520   2018-10-03
51521   2018-11-14
51522   2018-11-15
51523   2018-07-02
51524   2018-06-05
Name: date_posted, Length: 51525, dtype: datetime64[us]>


In [ ]:
# Verificando as infromações do dataframe novamente
print(car_data.info())

<class 'pandas.DataFrame'>
RangeIndex: 51525 entries, 0 to 51524
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   price         51525 non-null  int64         
 1   model_year    51525 non-null  int64         
 2   model         51525 non-null  str           
 3   condition     51525 non-null  str           
 4   cylinders     51525 non-null  int64         
 5   fuel          51525 non-null  str           
 6   odometer      51484 non-null  float64       
 7   transmission  51525 non-null  str           
 8   type          51525 non-null  str           
 9   paint_color   51525 non-null  str           
 10  is_4wd        51525 non-null  int64         
 11  date_posted   51525 non-null  datetime64[us]
 12  days_listed   51525 non-null  int64         
dtypes: datetime64[us](1), float64(1), int64(5), str(6)
memory usage: 7.2 MB
None


In [ ]:
# Visualizando os dados ausentes em 'odometer'
odometer_null = car_data[car_data['odometer'].isna()]
odometer_null

,price,model_year,model,condition,cylinders,fuel,odometer,transmission,type,paint_color,is_4wd,date_posted,days_listed
42,34900,2013,mercedes-benz benze sprinter 2500,excellent,6,diesel,NaN,automatic,van,black,0,2019-01-15,16
1642,34900,2013,mercedes-benz benze sprinter 2500,excellent,6,diesel,NaN,automatic,van,black,0,2018-12-04,36
2232,34900,2013,mercedes-benz benze sprinter 2500,excellent,6,diesel,NaN,automatic,van,black,0,2018-08-23,70
2731,34900,2013,mercedes-benz benze sprinter 2500,excellent,6,diesel,NaN,automatic,van,black,0,2019-04-12,31
4149,34900,2013,mercedes-benz benze sprinter 2500,excellent,6,diesel,NaN,automatic,van,black,0,2018-10-12,28
4681,34900,2013,mercedes-benz benze sprinter 2500,excellent,6,diesel,NaN,automatic,van,unknown,0,2018-10-02,32
5681,34900,2013,mercedes-benz benze sprinter 2500,excellent,6,diesel,NaN,automatic,van,black,0,2018-12-11,34
8975,34900,2013,mercedes-benz benze sprinter 2500,excellent,6,diesel,NaN,automatic,van,unknown,0,2018-09-24,45
10600,34900,2013,mercedes-benz benze sprinter 2500,excellent,6,diesel,NaN,automatic,van,black,0,2018-09-16,47
11541,34900,2013,mercedes-benz benze sprinter 2500,excellent,6,diesel,NaN,automatic,van,unknown,0,2018-05-28,24


In [ ]:

# Como são poucos dados e parece que não há dados para fazer a mediana por grupo, vou excluir esses dados do dataframe
car_data = car_data.dropna(subset=['odometer']).reset_index(drop=True)
print(len(car_data))


51484


In [ ]:
car_data.to_csv('vehicles_us_modify.csv', sep=',')

Agora nosso dataframe está limpo e manipulado para seguir para a visualização de dados no dashboard